In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import argparse
import os

In [ ]:
block_size = 64 # 32, 64, 128, 256
threshold = 4
visualflag = False
clean_path = "data/clean"
# data_path = "data/defocus"
data_path = "data/mobile"

In [ ]:
# Define detection function
def detect_blur_fft(image, size=60, threshold=10, vis=False):
    (h, w) = image.shape
    (cx, cy) = (int(w / 2.0), int(h / 2.0))
    fft = np.fft.fft2(image)
    fftShift = np.fft.fftshift(fft)

    if vis:
        magnitude = 20 * np.log(np.abs(fftShift))

        (fig, ax) = plt.subplots(1, 2,)

        ax[0].imshow(image, cmap="gray")
        ax[0].set_title("Input")
        ax[0].set_xticks([])
        ax[0].set_yticks([])

        ax[1].imshow(magnitude, cmap="gray")
        ax[1].set_title("Magnitude Spectrum")
        ax[1].set_xticks([])
        ax[1].set_yticks([])

        plt.show()

    fftShift[cy - size : cy + size, cx - size : cx + size] = 0
    fftShift = np.fft.ifftshift(fftShift)
    recon = np.fft.ifft2(fftShift)

    magnitude = 20 * np.log(np.abs(recon))
    mean = np.mean(magnitude)

    return (mean, mean <= threshold)

In [ ]:
# Define threshold function
def detect_threshold(image, size=60):

    (h, w) = image.shape
    (cx, cy) = (int(w / 2.0), int(h / 2.0))
    fft = np.fft.fft2(image)
    fftShift = np.fft.fftshift(fft)

    fftShift[cy - size : cy + size, cx - size : cx + size] = 0
    fftShift = np.fft.ifftshift(fftShift)
    recon = np.fft.ifft2(fftShift)

    magnitude = 20 * np.log(np.abs(recon))
    mean = np.mean(magnitude)

    return mean

In [ ]:
# Detect blury
results_file = 'outtxt/results_'+str(block_size)+'.txt'
with open(results_file, "a") as f:
    f.write("Image_Name, Mean, Blurry_Status\n")

# Detect threshold
threshold = 0
for root, sub_dirs, files in os.walk(clean_path):
    for file in files:

        print("\nDetect "+file+" using "+f'FFT_{block_size}')
        image = cv2.imread(os.path.join(root, file))
        h, w, r = image.shape
        gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray_img = gray_img[128:128+1024, 256:256+1536] # cropped image size
        gray = cv2.resize(gray_img, (0,0), fx=0.5, fy=0.5) 

        metrics = []
        for y in range(0, h, block_size):
            for x in range(0, w, block_size):
                block = gray[y:min(y + block_size, h), x:min(x + block_size, w)]
                if block.shape[0] == block_size and block.shape[1] == block_size:
                    mean= detect_threshold(block, size=60)
                    metrics.append(mean)
        threshold = np.mean(metrics)

        res = np.dstack([gray] * 3)
        color = (0, 255, 0)
        text = "Threshold ({:.4f})"
        text = text.format(threshold)
        cv2.putText(res, text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        print("Info: ", text, " ", file)
        cv2.imwrite("outimg/Det_"+f'FFT_{block_size}_'+file, res)
        print("Finish "+file+" using "+f'FFT_{block_size}'+'\n')

for root, sub_dirs, files in os.walk(data_path):
    for file in files:
        print("\nDetect "+file)
        orig = cv2.imread(os.path.join(root, file))
        h,w, r = orig.shape
        gray_img = cv2.cvtColor(orig, cv2.COLOR_BGR2GRAY)
        gray_img = gray_img[128:128+1024, 256:256+1536] # cropped image size
        gray = cv2.resize(gray_img, (0,0), fx=0.5, fy=0.5) 

        # mean, blurry = detect_blur_fft(
        #     gray, size=60, threshold=threshold, vis=visualflag > 0
        # )
        
        metrics = []
        for y in range(0, h, block_size):
            for x in range(0, w, block_size):
                block = gray[y:min(y + block_size, h), x:min(x + block_size, w)]
                if block.shape[0] == block_size and block.shape[1] == block_size:
                    mean, _ = detect_blur_fft(block, size=60, threshold=4, vis=False)
                    metrics.append(mean)
        result = np.mean(metrics)
        blurry = result < threshold

        image = np.dstack([gray] * 3)
        color = (0, 0, 255) if blurry else (0, 255, 0)
        text = "Blurry ({:.4f})" if blurry else "Not Blurry ({:.4f})"
        text = text.format(result)
        cv2.putText(image, text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        print("Info: ", text)

        # cv2.imshow("Output", image)
        # cv2.waitKey(0)
        print("Finish "+file+'\n')
        cv2.imwrite("outimg/Det_"+f'FFT_{block_size}_'+file, image)

        with open(results_file, "a") as f:
            f.write("{}, {:.4f}, {}\n".format(file, mean, "Blurry" if blurry else "Not Blurry"))